# Stage 2 — Luca Audit and Data Understanding

This Notebook provides **Stage 2 independently executed data checks** against the formal raw CSV. It supports a read-only audit of Luca Basanisi's reference treatment; it does not execute or copy the Luca Notebook.

Scope protections:

- Read `data/raw/UCI_Credit_Card.csv` without modifying or exporting it.
- Perform data understanding and treatment-verification checks only.
- Do not implement formal feature engineering, sampling, train/test splitting, cross-validation, threshold optimization, model selection, or model training.
- All displayed numbers below come from this Notebook's execution, not from Luca's saved outputs.

In [1]:
from pathlib import Path
import hashlib
import sys
import numpy as np
import pandas as pd
from IPython.display import display

launch_dir = Path.cwd().resolve()
search_roots = [launch_dir, launch_dir.parent]
project_root = next((p for p in search_roots if (p / 'data/raw/UCI_Credit_Card.csv').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Could not resolve project root from the current or parent directory.')
data_path = project_root / 'data/raw/UCI_Credit_Card.csv'

print('Stage 2 independently executed data check')
print(f'Python executable: {sys.executable}')
print(f'Python version: {sys.version.split()[0]}')
print(f'Launch directory: {launch_dir}')
print(f'Project root: {project_root}')
print(f'Raw CSV: {data_path}')

Stage 2 independently executed data check
Python executable: /Users/yu/2.工作/Risk Analysis Project/.venv/bin/python
Python version: 3.9.6
Launch directory: /Users/yu/2.工作/Risk Analysis Project
Project root: /Users/yu/2.工作/Risk Analysis Project
Raw CSV: /Users/yu/2.工作/Risk Analysis Project/data/raw/UCI_Credit_Card.csv


## Load and basic structure

The CSV is loaded once into memory. No write method is called.

In [2]:
df = pd.read_csv(data_path)
raw_sha256 = hashlib.sha256(data_path.read_bytes()).hexdigest()
print(f'Shape: {df.shape}')
print(f'Raw file bytes: {data_path.stat().st_size}')
print(f'Raw SHA-256: {raw_sha256}')
print('Columns in source order:')
print(df.columns.tolist())
dtype_table = df.dtypes.astype(str).rename('dtype').to_frame()
display(dtype_table)

Shape: (30000, 25)
Raw file bytes: 2862995
Raw SHA-256: a0f0ab49d6326671d6cd83be5c88dcf18007025fe9a53ecd699119c871176ca1
Columns in source order:
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default.payment.next.month']


,dtype
ID,int64
LIMIT_BAL,float64
SEX,int64
EDUCATION,int64
MARRIAGE,int64
AGE,int64
PAY_0,int64
PAY_2,int64
PAY_3,int64
PAY_4,int64


## Basic integrity checks

Checks cover nulls, exact duplicate rows, column-name duplication, numeric finiteness, ID existence and uniqueness, and target identity/distribution.

In [3]:
target_candidates = [c for c in df.columns if 'default' in c.lower()]
target_col = target_candidates[0] if len(target_candidates) == 1 else None
numeric = df.select_dtypes(include=[np.number])
integrity = pd.Series({
    'row_count': len(df),
    'column_count': df.shape[1],
    'total_null_cells': int(df.isna().sum().sum()),
    'columns_with_nulls': int((df.isna().sum() > 0).sum()),
    'exact_duplicate_rows': int(df.duplicated().sum()),
    'duplicate_column_names': int(df.columns.duplicated().sum()),
    'non_finite_numeric_cells': int((~np.isfinite(numeric.to_numpy())).sum()),
    'ID_present': 'ID' in df.columns,
    'ID_unique_values': int(df['ID'].nunique()) if 'ID' in df.columns else np.nan,
    'duplicate_ID_rows': int(df['ID'].duplicated().sum()) if 'ID' in df.columns else np.nan,
    'target_candidates': target_candidates,
    'target_selected': target_col,
})
display(integrity.to_frame('value'))
null_counts = df.isna().sum().rename('null_count').to_frame()
display(null_counts)
target_summary = pd.DataFrame({
    'count': df[target_col].value_counts(dropna=False).sort_index(),
    'proportion': df[target_col].value_counts(normalize=True, dropna=False).sort_index(),
})
display(target_summary)

,value
row_count,30000
column_count,25
total_null_cells,0
columns_with_nulls,0
exact_duplicate_rows,0
duplicate_column_names,0
non_finite_numeric_cells,0
ID_present,True
ID_unique_values,30000
duplicate_ID_rows,0


,null_count
ID,0
LIMIT_BAL,0
SEX,0
EDUCATION,0
MARRIAGE,0
AGE,0
PAY_0,0
PAY_2,0
PAY_3,0
PAY_4,0


,count,proportion
default.payment.next.month,,
0,23364,0.7788
1,6636,0.2212


## Demographics

Observed values are reported without recoding. UCI documents `SEX` values 1–2, `EDUCATION` values 1–4, and `MARRIAGE` values 1–3; therefore observed codes outside those documented sets are explicitly flagged for audit.

In [4]:
demographic_cols = ['SEX', 'EDUCATION', 'MARRIAGE', 'AGE']
documented_codes = {'SEX': {1, 2}, 'EDUCATION': {1, 2, 3, 4}, 'MARRIAGE': {1, 2, 3}}
demo_overview_rows = []
for col in demographic_cols:
    observed = sorted(df[col].dropna().unique().tolist())
    special = sorted(set(observed) - documented_codes[col]) if col in documented_codes else []
    demo_overview_rows.append({
        'variable': col,
        'missing': int(df[col].isna().sum()),
        'n_unique': int(df[col].nunique(dropna=True)),
        'min': df[col].min(),
        'max': df[col].max(),
        'observed_values': observed,
        'observed_outside_documented_codes': special,
    })
display(pd.DataFrame(demo_overview_rows).set_index('variable'))

for col in demographic_cols:
    counts = df[col].value_counts(dropna=False).sort_index()
    table = pd.DataFrame({'count': counts, 'percentage': counts / len(df) * 100})
    print(f'{col} frequency and percentage')
    display(table)

,missing,n_unique,min,max,observed_values,observed_outside_documented_codes
variable,,,,,,
SEX,0,2,1,2,"[1, 2]",[]
EDUCATION,0,7,0,6,"[0, 1, 2, 3, 4, 5, 6]","[0, 5, 6]"
MARRIAGE,0,4,0,3,"[0, 1, 2, 3]",[0]
AGE,0,56,21,79,"[21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 3...",[]


SEX frequency and percentage


,count,percentage
SEX,,
1,11888,39.626667
2,18112,60.373333


EDUCATION frequency and percentage


,count,percentage
EDUCATION,,
0,14,0.046667
1,10585,35.283333
2,14030,46.766667
3,4917,16.390000
4,123,0.410000
5,280,0.933333
6,51,0.170000


MARRIAGE frequency and percentage


,count,percentage
MARRIAGE,,
0,54,0.180000
1,13659,45.530000
2,15964,53.213333
3,323,1.076667


AGE frequency and percentage


,count,percentage
AGE,,
21,67,0.223333
22,560,1.866667
23,931,3.103333
24,1127,3.756667
25,1186,3.953333
26,1256,4.186667
27,1477,4.923333
28,1409,4.696667
29,1605,5.350000


In [5]:
print('AGE descriptive statistics')
display(df['AGE'].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).to_frame('AGE'))
for col in ['SEX', 'EDUCATION', 'MARRIAGE']:
    audit_table = pd.crosstab(df[col], df[target_col])
    audit_table['default_rate'] = df.groupby(col)[target_col].mean()
    print(f'{col} by target — audit-only crosstab')
    display(audit_table)

AGE descriptive statistics


,AGE
count,30000.000000
mean,35.485500
std,9.217904
min,21.000000
1%,22.000000
25%,28.000000
50%,34.000000
75%,41.000000
99%,60.000000
max,79.000000


SEX by target — audit-only crosstab


default.payment.next.month,0,1,default_rate
SEX,,,
1,9015,2873,0.241672
2,14349,3763,0.207763


EDUCATION by target — audit-only crosstab


default.payment.next.month,0,1,default_rate
EDUCATION,,,
0,14,0,0.000000
1,8549,2036,0.192348
2,10700,3330,0.237349
3,3680,1237,0.251576
4,116,7,0.056911
5,262,18,0.064286
6,43,8,0.156863


MARRIAGE by target — audit-only crosstab


default.payment.next.month,0,1,default_rate
MARRIAGE,,,
0,49,5,0.092593
1,10453,3206,0.234717
2,12623,3341,0.209283
3,239,84,0.260062


## Credit exposure: `LIMIT_BAL`

Only descriptive distribution checks are performed; no binning or transformation is searched or created.

In [6]:
limit_summary = df['LIMIT_BAL'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame('LIMIT_BAL')
limit_extra = pd.Series({
    'missing': int(df['LIMIT_BAL'].isna().sum()),
    'zero_count': int((df['LIMIT_BAL'] == 0).sum()),
    'negative_count': int((df['LIMIT_BAL'] < 0).sum()),
    'unique_values': int(df['LIMIT_BAL'].nunique()),
}, name='LIMIT_BAL')
display(limit_summary)
display(limit_extra.to_frame())

,LIMIT_BAL
count,30000.000000
mean,167484.322667
std,129747.661567
min,10000.000000
1%,10000.000000
5%,20000.000000
25%,50000.000000
50%,140000.000000
75%,240000.000000
95%,430000.000000


,LIMIT_BAL
missing,0
zero_count,0
negative_count,0
unique_values,81


## Repayment / delinquency status deep dive

Status columns are identified explicitly so `PAY_AMT*` is not mixed into the group. The tables preserve raw codes. A temporary audit mapping reproduces Luca's final collapse (`code <= 0 → 0`, `code > 0 → 1`) only to quantify information compression; it is not added to the data and is not a project feature.

In [7]:
status_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
assert all(c in df.columns for c in status_cols)
status_overview = pd.DataFrame({
    'missing': df[status_cols].isna().sum(),
    'min': df[status_cols].min(),
    'max': df[status_cols].max(),
    'n_unique': df[status_cols].nunique(),
    'unique_codes': [sorted(df[c].dropna().unique().tolist()) for c in status_cols],
})
display(status_overview)

status_counts = pd.concat({c: df[c].value_counts().sort_index() for c in status_cols}, axis=1).fillna(0).astype(int)
status_percent = status_counts / len(df) * 100
print('Raw code counts by status column')
display(status_counts)
print('Raw code percentages by status column')
display(status_percent.round(4))

,missing,min,max,n_unique,unique_codes
PAY_0,0,-2,8,11,"[-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]"
PAY_2,0,-2,8,11,"[-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]"
PAY_3,0,-2,8,11,"[-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]"
PAY_4,0,-2,8,11,"[-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]"
PAY_5,0,-2,8,10,"[-2, -1, 0, 2, 3, 4, 5, 6, 7, 8]"
PAY_6,0,-2,8,10,"[-2, -1, 0, 2, 3, 4, 5, 6, 7, 8]"


Raw code counts by status column


,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
-2,2759,3782,4085,4348,4546,4895
-1,5686,6050,5938,5687,5539,5740
0,14737,15730,15764,16455,16947,16286
1,3688,28,4,2,0,0
2,2667,3927,3819,3159,2626,2766
3,322,326,240,180,178,184
4,76,99,76,69,84,49
5,26,25,21,35,17,13
6,11,12,23,5,4,19
7,9,20,27,58,58,46


Raw code percentages by status column


,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
-2,9.1967,12.6067,13.6167,14.4933,15.1533,16.3167
-1,18.9533,20.1667,19.7933,18.9567,18.4633,19.1333
0,49.1233,52.4333,52.5467,54.8500,56.4900,54.2867
1,12.2933,0.0933,0.0133,0.0067,0.0000,0.0000
2,8.8900,13.0900,12.7300,10.5300,8.7533,9.2200
3,1.0733,1.0867,0.8000,0.6000,0.5933,0.6133
4,0.2533,0.3300,0.2533,0.2300,0.2800,0.1633
5,0.0867,0.0833,0.0700,0.1167,0.0567,0.0433
6,0.0367,0.0400,0.0767,0.0167,0.0133,0.0633
7,0.0300,0.0667,0.0900,0.1933,0.1933,0.1533


In [8]:
default_rate_by_status = pd.concat(
    {c: df.groupby(c)[target_col].agg(['count', 'sum', 'mean']) for c in status_cols},
    names=['status_variable', 'raw_code'],
)
default_rate_by_status = default_rate_by_status.rename(columns={'sum': 'defaults', 'mean': 'default_rate'})
display(default_rate_by_status)

collapsed = (df[status_cols] > 0).astype('int8')  # audit-only; not assigned to df
compression = pd.DataFrame({
    'raw_levels': df[status_cols].nunique(),
    'levels_after_Luca_final_binary_mapping': collapsed.nunique(),
    'rows_mapped_to_0': (collapsed == 0).sum(),
    'rows_mapped_to_1': (collapsed == 1).sum(),
    'rows_with_raw_severity_gt_1': (df[status_cols] > 1).sum(),
})
display(compression)
trajectory_summary = pd.Series({
    'unique_raw_six_month_trajectories': int(df[status_cols].drop_duplicates().shape[0]),
    'unique_binary_six_month_trajectories': int(collapsed.drop_duplicates().shape[0]),
    'rows_with_any_positive_delay': int((df[status_cols] > 0).any(axis=1).sum()),
    'rows_with_any_severity_above_one': int((df[status_cols] > 1).any(axis=1).sum()),
    'rows_with_any_minus_two': int((df[status_cols] == -2).any(axis=1).sum()),
    'rows_with_any_minus_one': int((df[status_cols] == -1).any(axis=1).sum()),
    'rows_with_any_zero': int((df[status_cols] == 0).any(axis=1).sum()),
}, name='audit_value')
display(trajectory_summary.to_frame())

count  defaults  default_rate
status_variable raw_code                               
PAY_0           -2         2759       365      0.132294
                -1         5686       954      0.167781
                 0        14737      1888      0.128113
                 1         3688      1252      0.339479
                 2         2667      1844      0.691414
...                         ...       ...           ...
PAY_6            4           49        31      0.632653
                 5           13         7      0.538462
                 6           19        14      0.736842
                 7           46        38      0.826087
                 8            2         2      1.000000

[64 rows x 3 columns]

,raw_levels,levels_after_Luca_final_binary_mapping,rows_mapped_to_0,rows_mapped_to_1,rows_with_raw_severity_gt_1
PAY_0,11,2,23182,6818,3130
PAY_2,11,2,25562,4438,4410
PAY_3,11,2,25787,4213,4209
PAY_4,11,2,26490,3510,3508
PAY_5,10,2,27032,2968,2968
PAY_6,10,2,26921,3079,3079


,audit_value
unique_raw_six_month_trajectories,1106
unique_binary_six_month_trajectories,62
rows_with_any_positive_delay,10069
rows_with_any_severity_above_one,8380
rows_with_any_minus_two,6561
rows_with_any_minus_one,10228
rows_with_any_zero,21173


## Billing and payment history

The audit records missingness, scale, quantiles, zeros, and negatives. It does not create utilization, payment ratios, trends, volatility, or other formal risk features.

In [9]:
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
payment_cols = [f'PAY_AMT{i}' for i in range(1, 7)]

def history_summary(columns):
    rows = []
    for col in columns:
        s = df[col]
        rows.append({
            'variable': col,
            'count': int(s.count()),
            'missing': int(s.isna().sum()),
            'mean': s.mean(),
            'std': s.std(),
            'min': s.min(),
            'p01': s.quantile(0.01),
            'p25': s.quantile(0.25),
            'median': s.median(),
            'p75': s.quantile(0.75),
            'p99': s.quantile(0.99),
            'max': s.max(),
            'zero_count': int((s == 0).sum()),
            'zero_percentage': (s == 0).mean() * 100,
            'negative_count': int((s < 0).sum()),
            'negative_percentage': (s < 0).mean() * 100,
        })
    return pd.DataFrame(rows).set_index('variable')

print('Billing history summary')
display(history_summary(bill_cols))
print('Payment history summary')
display(history_summary(payment_cols))
group_behavior = pd.Series({
    'rows_all_six_bill_amounts_zero': int((df[bill_cols] == 0).all(axis=1).sum()),
    'rows_any_bill_amount_negative': int((df[bill_cols] < 0).any(axis=1).sum()),
    'rows_all_six_payment_amounts_zero': int((df[payment_cols] == 0).all(axis=1).sum()),
    'rows_any_payment_amount_negative': int((df[payment_cols] < 0).any(axis=1).sum()),
}, name='audit_value')
display(group_behavior.to_frame())

Billing history summary


,count,missing,mean,std,min,p01,p25,median,p75,p99,max,zero_count,zero_percentage,negative_count,negative_percentage
variable,,,,,,,,,,,,,,,
BILL_AMT1,30000,0,51223.330900,73635.860576,-165580.0,-81.00,3558.75,22381.5,67091.00,350110.68,964511.0,2008,6.693333,590,1.966667
BILL_AMT2,30000,0,49179.075167,71173.768783,-69777.0,-200.00,2984.75,21200.0,64006.25,337495.28,983931.0,2506,8.353333,669,2.230000
BILL_AMT3,30000,0,47013.154800,69349.387427,-157264.0,-200.00,2666.25,20088.5,60164.75,325030.39,1664089.0,2870,9.566667,655,2.183333
BILL_AMT4,30000,0,43262.948967,64332.856134,-170000.0,-212.02,2326.75,19052.0,54506.00,304997.27,891586.0,3195,10.650000,675,2.250000
BILL_AMT5,30000,0,40311.400967,60797.155770,-81334.0,-232.01,1763.00,18104.5,50190.50,285868.33,927171.0,3506,11.686667,655,2.183333
BILL_AMT6,30000,0,38871.760400,59554.107537,-339603.0,-331.03,1256.00,17071.0,49198.25,279505.06,961664.0,4020,13.400000,688,2.293333


Payment history summary


,count,missing,mean,std,min,p01,p25,median,p75,p99,max,zero_count,zero_percentage,negative_count,negative_percentage
variable,,,,,,,,,,,,,,,
PAY_AMT1,30000,0,5663.580500,16563.280354,0.0,0.0,1000.00,2100.0,5006.00,66522.18,873552.0,5249,17.496667,0,0.0
PAY_AMT2,30000,0,5921.163500,23040.870402,0.0,0.0,833.00,2009.0,5000.00,76651.02,1684259.0,5396,17.986667,0,0.0
PAY_AMT3,30000,0,5225.681500,17606.961470,0.0,0.0,390.00,1800.0,4505.00,70000.00,896040.0,5968,19.893333,0,0.0
PAY_AMT4,30000,0,4826.076867,15666.159744,0.0,0.0,296.00,1500.0,4013.25,67054.44,621000.0,6408,21.360000,0,0.0
PAY_AMT5,30000,0,4799.387633,15278.305679,0.0,0.0,252.50,1500.0,4031.50,65607.56,426529.0,6703,22.343333,0,0.0
PAY_AMT6,30000,0,5215.502567,17777.465775,0.0,0.0,117.75,1500.0,4000.00,82619.05,528666.0,7173,23.910000,0,0.0


,audit_value
rows_all_six_bill_amounts_zero,866
rows_any_bill_amount_negative,1930
rows_all_six_payment_amounts_zero,1432
rows_any_payment_amount_negative,0


## Target and final reproducibility assertions

The class distribution is described only. It is not changed and no imbalance experiment is run.

In [10]:
target_values = sorted(df[target_col].dropna().unique().tolist())
target_counts = df[target_col].value_counts().sort_index()
target_table = pd.DataFrame({
    'count': target_counts,
    'proportion': target_counts / len(df),
    'percentage': target_counts / len(df) * 100,
})
display(target_table)
print(f'Target unique values: {target_values}')
print(f'Majority/minority count ratio: {target_counts.max() / target_counts.min():.6f}')
print('Imbalance note: class 1 is the minority; no sampling or class weighting is performed here.')

assert df.shape == (30000, 25)
assert target_col == 'default.payment.next.month'
assert target_values == [0, 1]
assert int(df.isna().sum().sum()) == 0
assert int(df['ID'].nunique()) == len(df)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == raw_sha256
print('All Stage 2 data-understanding assertions passed.')
print('No processed dataset was written; no model or sampling procedure was executed.')

,count,proportion,percentage
default.payment.next.month,,,
0,23364,0.7788,77.88
1,6636,0.2212,22.12


Target unique values: [0, 1]
Majority/minority count ratio: 3.520796
Imbalance note: class 1 is the minority; no sampling or class weighting is performed here.
All Stage 2 data-understanding assertions passed.
No processed dataset was written; no model or sampling procedure was executed.
